# Identify multi-hazard events for a specific country

In [2]:
import os
import pandas as pd
import glob
from pandas.tseries.offsets import DateOffset

In [3]:
path = '/gpfs/work4/0/FWC2/Spatial_dependence/Disease_outbreak/global'
look_window = 1
disease_type = 'waterborne'
files = glob.glob(os.path.join(path,  'data', 'eca_binary','multi_hazard', f'{disease_type}', 'Natural Hazard', f'df_eca_Natural Hazard_*.csv'))
country_list = [
    os.path.basename(f).replace('df_eca_Natural Hazard_', '').replace('.csv', '')
    for f in files
]

In [4]:
len(country_list)

49

In [5]:
for COUNTRY in country_list:
    df_hazards = pd.read_csv(os.path.join(path, 'data', 'eca_binary','multi_hazard', f'{disease_type}', 'Natural Hazard',f'df_eca_Natural Hazard_{COUNTRY}.csv'))
    df_hazards["Start date"] = pd.to_datetime(df_hazards["Start date"], errors="coerce").dt.tz_localize(None)
    df_epidemics = pd.read_csv(os.path.join(path, 'data', 'eca_binary','multi_hazard', f'{disease_type}', 'Natural Hazard',f'df_eca_Epidemic_{COUNTRY}.csv'))
    df = df_hazards.copy()
    events = []

    for loc, group in df.groupby("Location"):

        group = group.sort_values("Start date").reset_index(drop=True)

        current_cluster = []
        current_end = None

        for i in range(len(group)):
            row = group.iloc[i]

            start = row["Start date"]
            end = start + DateOffset(months=look_window)

            if current_end is None:
                current_cluster = [row]
                current_end = end

            elif start <= current_end:
                # same cluster
                current_cluster.append(row)
                current_end = max(current_end, end)

            else:
                events.append(current_cluster)

                # start new cluster
                current_cluster = [row]
                current_end = end

        # last cluster (also include single)
        if current_cluster:
            events.append(current_cluster)
    
    records = []

    for cluster in events:
        df_c = pd.DataFrame(cluster)
        dates = df_c["Start date"]
        multi = (len(df_c)>1 or df_c["Associated Types"].notna().all())

        records.append({
            "Location": df_c['Location'].iloc[0],
            "Start date": pd.to_datetime(
                            dates.min(),
                            format='%d/%m/%Y',
                            utc=True
                        ),
            "End date": pd.to_datetime(
                            dates.max(),
                            format='%d/%m/%Y',
                            utc=True
                        ),
            "N_hazards": len(df_c),
            "Multi": multi,
            "DisNo.": ", ".join(df_c["DisNo."]),
            "Hazards_subgroup": ", ".join(df_c["Disaster Subgroup"]),
            "Hazards_type": ", ".join(df_c["Disaster Type"]),
            "Associated_types": ", ".join(df_c["Associated Types"].astype(str))
        })

    df_events = pd.DataFrame(records)
    out_dir = os.path.join(path,'data','eca_binary','multi_hazard','waterborne','hazard_all')
    os.makedirs(out_dir, exist_ok=True)
    df_events.to_csv(os.path.join(out_dir,f'df_multi_hazard_{COUNTRY}.csv'), index=False)

    # separate into multi and single hazard event sets
    df_multi = df_events[df_events["Multi"] == True]
    df_single = df_events[df_events["Multi"] == False]

    out_dir = os.path.join(path,'data','eca_binary','multi_hazard','waterborne','multi_hazard')
    os.makedirs(out_dir, exist_ok=True)
    df_multi.to_csv(os.path.join(out_dir,f'df_multi_hazard_{COUNTRY}.csv'), index=False)

    out_dir = os.path.join(path,'data','eca_binary','multi_hazard','waterborne','single_hazard')
    os.makedirs(out_dir, exist_ok=True)
    df_single.to_csv(os.path.join(out_dir,f'df_single_hazard_{COUNTRY}.csv'), index=False)

In [33]:
cluster = events[1]
df_c = pd.DataFrame(cluster)
df_c

,Start date,DisNo.,Disaster Subgroup,Disaster Type,Associated Types,Location,Natural Hazard
1,2003-07-01,2003-0394-CHN,Meteorological,Storm,Rain,Anhui,True


np.True_

In [12]:
look_window = 3 # months
multi_events = []
for location, group in df_hazards.groupby("Location"):
    dates = group["Start date"].values

    for i in range(len(group)):
        current_date = dates[i]
        end_window = current_date + DateOffset(months=look_window)

        # Count events in window (including itself)
        count = ((group["Start date"] >= current_date) & 
                 (group["Start date"] <= end_window)).sum()

        if count >= 2:
            multi_events.append(group.iloc[i])

In [13]:
df_multi = pd.DataFrame(multi_events)
df_multi

,Start date,DisNo.,Disaster Subgroup,Disaster Type,Location,Natural Hazard
310,2000-08-01,2000-0534-IND,Hydrological,Flood,Andhra Pradesh,True
311,2000-08-01,2000-0445-IND,Hydrological,Flood,Andhra Pradesh,True
372,2005-05-01,2005-0290-IND,Meteorological,Storm,Andhra Pradesh,True
383,2005-07-01,2005-0399-IND,Hydrological,Flood,Andhra Pradesh,True
397,2006-04-01,2006-0207-IND,Hydrological,Flood,Andhra Pradesh,True
...,...,...,...,...,...,...
391,2005-10-01,2005-0599-IND,Hydrological,Flood,West Bengal,True
393,2005-10-01,2005-0627-IND,Hydrological,Flood,West Bengal,True
431,2007-09-01,2007-0467-IND,Hydrological,Flood,West Bengal,True
603,2018-08-01,2018-0295-IND,Hydrological,Flood,West Bengal,True
